# Stage 1: Non-Instruction Fine-Tuning
**Goal:** Adapt base model to finance domain language before instruction tuning.

## Step 1 — Install dependencies

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes

## Step 2 — Imports

In [ ]:
import torch
from unsloth import FastLanguageModel
from datasets import Dataset
from trl import SFTTrainer
from transformers import TrainingArguments
import re

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Step 3 — Configuration

In [ ]:
# ── Model config ──
MODEL_NAME    = "unsloth/Llama-3.2-1B"
MAX_SEQ_LEN   = 512
LOAD_IN_4BIT  = True   # QLoRA — keeps VRAM under 8GB

# ── LoRA config ──
LORA_R        = 16
LORA_ALPHA    = 16
LORA_DROPOUT  = 0.05
TARGET_MODS   = ["q_proj", "k_proj", "v_proj", "o_proj",
                 "up_proj", "down_proj", "gate_proj"]

# ── Training config ──
OUTPUT_DIR    = "./outputs/stage1_non_instruction"
EPOCHS        = 3
BATCH_SIZE    = 2
GRAD_ACCUM    = 4
LR            = 2e-4

## Step 4 — Load base model

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name      = MODEL_NAME,
    max_seq_length  = MAX_SEQ_LEN,
    dtype           = None,          # Auto-detect (bfloat16 on A100, float16 on T4)
    load_in_4bit    = LOAD_IN_4BIT,
)
print("Base model loaded successfully")

## Step 5 — Apply LoRA

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r                    = LORA_R,
    target_modules       = TARGET_MODS,
    lora_alpha           = LORA_ALPHA,
    lora_dropout         = LORA_DROPOUT,
    bias                 = "none",
    use_gradient_checkpointing = "unsloth",
    random_state         = 42,
)
model.print_trainable_parameters()

## Step 6 — Load and prepare raw finance text

In [ ]:
# Load raw text from file
with open("data/non_instruction_data.txt", "r") as f:
    raw_text = f.read()

# Remove comment lines starting with #
lines = [l for l in raw_text.split("\n") if not l.startswith("#")]
raw_text = "\n".join(lines)

# Split into paragraphs and filter empty ones
paragraphs = [p.strip() for p in raw_text.split("\n\n") if len(p.strip()) > 50]
print(f"Total paragraphs loaded: {len(paragraphs)}")
print("\nSample paragraph:")
print(paragraphs[0][:200])

## Step 7 — Tokenize and chunk text

In [ ]:
# Wrap paragraphs in EOS token for causal LM training
EOS = tokenizer.eos_token

def chunk_text(paragraphs, tokenizer, max_len=MAX_SEQ_LEN):
    chunks = []
    current_chunk = ""
    for para in paragraphs:
        candidate = current_chunk + " " + para if current_chunk else para
        toks = tokenizer(candidate, return_tensors="pt")["input_ids"].shape[1]
        if toks <= max_len:
            current_chunk = candidate
        else:
            if current_chunk:
                chunks.append(current_chunk + EOS)
            current_chunk = para
    if current_chunk:
        chunks.append(current_chunk + EOS)
    return chunks

chunks = chunk_text(paragraphs, tokenizer)
print(f"Total training chunks: {len(chunks)}")

dataset = Dataset.from_dict({"text": chunks})
print(dataset)

## Step 8 — Train

In [ ]:
trainer = SFTTrainer(
    model            = model,
    tokenizer        = tokenizer,
    train_dataset    = dataset,
    dataset_text_field = "text",
    max_seq_length   = MAX_SEQ_LEN,
    args = TrainingArguments(
        output_dir              = OUTPUT_DIR,
        num_train_epochs        = EPOCHS,
        per_device_train_batch_size = BATCH_SIZE,
        gradient_accumulation_steps = GRAD_ACCUM,
        learning_rate           = LR,
        fp16                    = not torch.cuda.is_bf16_supported(),
        bf16                    = torch.cuda.is_bf16_supported(),
        logging_steps           = 10,
        save_strategy           = "epoch",       # Save after every epoch
        warmup_ratio            = 0.1,
        lr_scheduler_type       = "cosine",
        report_to               = "none",
    ),
)

print("Starting non-instruction fine-tuning...")
trainer_stats = trainer.train()
print(f"Training complete. Final loss: {trainer_stats.training_loss:.4f}")

## Step 9 — Save adapter

In [ ]:
ADAPTER_PATH = "./outputs/stage1_adapter"
model.save_pretrained(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)
print(f"Stage 1 adapter saved to: {ADAPTER_PATH}")

# Also save to Google Drive if available
try:
    from google.colab import drive
    drive.mount("/content/drive")
    model.save_pretrained("/content/drive/MyDrive/finance_ft/stage1_adapter")
    print("Adapter also saved to Google Drive")
except:
    print("Not on Colab — skipping Drive save")

## Step 10 — Test model after non-instruction fine-tuning

In [ ]:
FastLanguageModel.for_inference(model)

test_prompts = [
    "A mutual fund is",
    "The Reserve Bank of India",
    "Systematic Investment Plan",
]

for prompt in test_prompts:
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(
        **inputs,
        max_new_tokens = 100,
        temperature    = 0.7,
        do_sample      = True,
        pad_token_id   = tokenizer.eos_token_id,
    )
    generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"Prompt: {prompt}")
    print(f"Output: {generated}")
    print("-" * 60)